<div style="text-align: center;">
    <h1> </font> <font color = #DC143C>Analysis of Risk Passenger Rule Generation for Individual Flights</h1> </font>
</div>

<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>01- Importing Required Libraries</h3> </font>
</div>

In [1]:
import numpy as np
import pandas as pd

import os
import pickle

from datetime import datetime

pd.options.display.max_columns = None
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce

from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans  

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules
import ast

import gym
from gym import spaces
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

import warnings
warnings.filterwarnings("ignore")

<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>02- Loading the Raw Staging dataset</h3> </font>
</div>

In [2]:
# Load data from the pickle file

def load_from_pickle(filename):
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    return data

data = load_from_pickle('STG_Data.pkl')
data

,flightDetails__id,flightDetails_flightNumber,flightDetails_departureDate,flightDetails_origin,flightDetails_destination,flightDetails_arrivalTime,flightDetails_depTime,flightDetails_noOfSeatOfFlight,flightDetails_noOfPassengers,flightDetails_aircraftType,flightDetails_seatDetailsList,_id,flightNumber,dataSource,uniqueFlightKey,type,givenName,lastName,isFoundPnr,nationality,gender,isVerified,residenceCountry,dependent,crew,transit,portOfEmbark,portOfDisembark,travellerReferenceId,travellerReferenceType,bookingDate,paxCount,seatNumber,dob,appDetails_givenName,appDetails_surname,appDetails_lastName,appDetails_gender,appDetails_dob,reservation_firstName,reservation_lastName,reservation_gender,paidParty_cardNumber,paidParty_expirationDateAsString,paidParty_expirationDate,paidParty_cardType,paidParty_payAmount,paidParty_paidCurrency,paidParty_paidDate,paidParty_cardHolderName,paidParty_terminalId,paidParty_paymentId,paidParty_authorizationCode,paidParty_merchantId,paidParty_payCountry,travelData_paxTypes,travelData_reservationReferenceType,travelData_reservationDate,travelData_noOfCheckingLuggage,travelData_reservationReferenceNumber,travelData_seatNumber,travelDocument_docType,travelDocument_docNo,travelDocument_expiryDate,travelDocument_issueCountry,firstTransitPorts,noOfCheckingLuggage,visaNo,visaExpiryDate,arrivalDateTime,depDateTime,fileId,apsHitPax_stop,apsHitPax_watch,apsHitPax_partial,apsHitPax_re,apsHitPax_cl,apsHitPax_visa,apsHitPax_link,apsHitPax_nsc,apsHitPax_noHit,apsHitPax_td,apsHitPax_tt,apsHitPax_interpol,apsHitPax_isBookingRefHit,apsHitPax_travelHistory,apsHitPax_isCrew,apsHitPax_isArriving,apsHitPax_isTransit,apsHitPax_isCleared,apsHitPax_hitRuleCodes,apsHitPax_ruleEngineValidation_type,apsHitPax_ruleEngineValidation_status,apsHitPax_ruleEngineValidation_validationData,apsHitPax_ruleEngineValidation_message,apsHitPax_clValidation_type,apsHitPax_clValidation_status,apsHitPax_clValidation_hitLevel,apsHitPax_clValidation_hitTypes,apsHitPax_bookingRefValidationData,apsHitPax_bookingRefNo,apsHitPax_stopValidation_type,apsHitPax_stopValidation_status,apsHitPax_stopValidation_hitLevel,apsHitPax_stopValidation_hitTypes,apsHitPax_stopValidation_validationData,apsHitPax_stopValidation_message,apsHitPax_clValidation_validationData,apsHitPax_clValidation_message,apsHitPax_stopValidationData,action,secondTransitPort,apsHitPax_ruleEngineValidationData,apsHitPax_watchValidation_type,apsHitPax_watchValidation_status,apsHitPax_watchValidation_hitLevel,apsHitPax_watchValidation_hitTypes,apsHitPax_watchValidation_validationData,apsHitPax_watchValidation_message,apsHitPax_watchValidationData,apsHitPax_bookingRefValidation_type,apsHitPax_bookingRefValidation_status,apsHitPax_nscValidation_type,apsHitPax_nscValidation_status,apsHitPax_nscValidation_message,apsHitPax_visaValidation_type,apsHitPax_visaValidation_status,apsHitPax_visaValidation_validationData,apsHitPax_visaValidation_message,apsHitPax_visaValidationData,apsHitPax_bookingRefValidation_validationData,apsHitPax_bookingRefValidation_message,apsHitPax_partialValidation_type,apsHitPax_partialValidation_status,apsHitPax_partialValidation_hitLevel,apsHitPax_partialValidation_hitTypes,apsHitPax_partialValidation_validationData,apsHitPax_partialValidation_message,apsHitPax_partialValidationData,apsHitPax__id,actionedOn,actionedBy,remarks
0,UL521-2024-03-20T19:45-CMB-KUL,UL521,2024-03-20T19:45,CMB,KUL,2024-03-20 14:15:00,2024-03-20 14:15:00,337,337,Boeing 777-300ER(77W) Three Class,"[{'_id': '567950436', 'seatClass': 'FIRST', 's...",N5330233-LKA-UL521-2024-03-20T19:45-CMB-KUL,UL521,PNR_FILE,UL521-2024-03-20T19:45-CMB-KUL,LOCAL,VIGNARAJAH,MANOHARAN,True,LKA,MALE,False,LKA,False,False,False,CMB,KUL,DSQAUZ,AVF,2024-01-05,1,1A,1975-09-03,FATHIMA,OOI,OOI,Male,1999-11-25,VIGNARAJAH,MANOHARAN,MALE,23118281,2024-01-05,2024-01-04 18:30:00,CARD,343,USD,2023-01-04 18:30:00,VIGNARAJAH MANOHARAN,70271,97059,877,41429,LKA,PAX,AVF,2024-01-05,1,DSQAUZ,1A,P,N5330233,2029-04-15,LKA,SGP,1,VP2023048,20

<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>03- Data Preprocessing</h3> </font>
</div>

This data_preprocessing function filters a dataset to include only rows where the 'apsHitPax_isArriving' column is True and selects specific columns for further processing. 

It converts various date columns to datetime format and sorts the data by 'flightDetails_arrivalTime', filtering out entries before March 20, 2024. 

The function then separates the 'flightDetails_arrivalTime' into 'arrival_date' and 'arrival_time' columns, adds a 'arrival_month_year' column, and converts 'travelData_noOfCheckingLuggage_' to numeric. 

It creates a unique identifier for each passenger by combining 'travelDocument_docNo' and 'nationality' and maps these to a numerical 'passenger_id'. Finally, the function returns the cleaned and processed dataset.

In [3]:
def data_preprocessing(data):
    
    # Filter entries where apsHitPax_isArriving is True
    data = data[data['apsHitPax_isArriving'] == True]
    
    # Select the desired columns
    selected_columns = [
        'flightDetails__id', 'flightDetails_flightNumber', 'flightDetails_arrivalTime', 'type', 'givenName', 'lastName',
        'nationality', 'gender', 'travelDocument_docNo', 'travelDocument_expiryDate', 'travelDocument_issueCountry',
        'firstTransitPorts', 'dob', 'residenceCountry', 'dependent', 'crew', 'transit', 'portOfEmbark', 'portOfDisembark',
        'travellerReferenceId', 'travellerReferenceType', 'bookingDate', 'paxCount', 'reservation_firstName',
        'reservation_lastName', 'reservation_gender', 'travelData_reservationDate', 'travelData_noOfCheckingLuggage',
        'paidParty_cardNumber', 'paidParty_expirationDate', 'paidParty_cardType', 'paidParty_paidCurrency',
        'paidParty_paidDate', 'paidParty_cardHolderName', 'paidParty_payCountry', 'travelData_paxTypes', 'visaNo',
        'visaExpiryDate'
    ]
    data = data[selected_columns]
        
    # Convert date columns to datetime
    date_columns = ['dob', 'travelDocument_expiryDate', 'visaExpiryDate', 'bookingDate', 'travelData_reservationDate']
    for col in date_columns:
        data[col] = pd.to_datetime(data[col])

    # Sort the DataFrame by the arrival time column
    data = data.sort_values(by='flightDetails_arrivalTime').reset_index(drop=True)
    
    # Filter out rows where 'flightDetails_arrivalTime' is before March 20, 2024
    data = data[data['flightDetails_arrivalTime'] >= '2024-03-20']
    
    # Reset index to have a sequential range starting from 0
    data = data.reset_index(drop=True)
    
    # Extract date and time into separate columns
    data['arrival_date'] = data['flightDetails_arrivalTime'].dt.date
    data['arrival_date'] = pd.to_datetime(data['arrival_date'])
    data['arrival_time'] = pd.to_datetime(data['flightDetails_arrivalTime']).dt.time.astype(str)
    data['arrival_time'] = pd.to_datetime(data['arrival_time'])
    
    # Extract month and year
    data['arrival_month_year'] = data['arrival_date'].dt.to_period('M')
    
    # Convert 'travelData_noOfCheckingLuggage' to numeric
    data['travelData_noOfCheckingLuggage'] = pd.to_numeric(data['travelData_noOfCheckingLuggage'], errors='coerce')
    
    # Create unique ID and map to numerical ID
    data['unique_id'] = data['travelDocument_docNo'].astype(str) + '_' + data['nationality']
    id_mapping = {id: idx for idx, id in enumerate(data['unique_id'].unique())}
    data['passenger_id'] = data['unique_id'].map(id_mapping)
    
    return data

In [4]:
data= data_preprocessing(data)
data

,flightDetails__id,flightDetails_flightNumber,flightDetails_arrivalTime,type,givenName,lastName,nationality,gender,travelDocument_docNo,travelDocument_expiryDate,travelDocument_issueCountry,firstTransitPorts,dob,residenceCountry,dependent,crew,transit,portOfEmbark,portOfDisembark,travellerReferenceId,travellerReferenceType,bookingDate,paxCount,reservation_firstName,reservation_lastName,reservation_gender,travelData_reservationDate,travelData_noOfCheckingLuggage,paidParty_cardNumber,paidParty_expirationDate,paidParty_cardType,paidParty_paidCurrency,paidParty_paidDate,paidParty_cardHolderName,paidParty_payCountry,travelData_paxTypes,visaNo,visaExpiryDate,arrival_date,arrival_time,arrival_month_year,unique_id,passenger_id
0,UL521-2024-03-20T19:45-CMB-KUL,UL521,2024-03-20 14:15:00,FOREIGNER,ERIC,BIN ABAS,FJI,MALE,X3630553,2030-08-22,FJI,NaN,1975-08-16,FJI,False,False,False,CMB,KUL,QITSZH,AVF,2023-03-08,1,ERIC,BIN ABAS,MALE,2023-03-08,1,42899883,2023-03-07 18:30:00,CARD,USD,2023-03-07 18:30:00,ERIC BIN ABAS,LKA,PAX,VP2023461,2025-07-14,2024-03-20,2025-01-31 14:15:00,2024-03,X3630553_FJI,0
1,UL521-2024-03-20T19:45-CMB-KUL,UL521,2024-03-20 14:15:00,FOREIGNER,FAISAL,LEO,CHN,FEMALE,Y1526703,2034-09-04,CHN,NaN,1994-06-20,CHN,False,False,False,CMB,KUL,RQTJOK,AVF,2023-03-13,1,FAISAL,LEO,FEMALE,2023-03-13,3,68642212,2023-03-12 18:30:00,CARD,USD,2023-03-12 18:30:00,FAISAL LEO,LKA,PAX,VP2023109,2025-05-15,2024-03-20,2025-01-31 14:15:00,2024-03,Y1526703_CHN,1
2,UL521-2024-03-20T19:45-CMB-KUL,UL521,2024-03-20 14:15:00,FOREIGNER,DIYA,BINTI MAMAT,GBR,MALE,K2662922,2024-04-11,GBR,NaN,1982-09-23,GBR,False,False,False,CMB,KUL,IVRAUK,AVF,2023-03-02,1,DIYA,BINTI MAMAT,MALE,2023-03-02,2,59588240,2023-03-01 18:30:00,CARD,USD,2023-03-01 18:30:00,DIYA BINTI MAMAT,LKA,PAX,VP2023729,2025-08-02,2024-03-20,2025-01-31 14:15:00,2024-03,K2662922_GBR,2
3,UL521-2024-03-20T19:45-CMB-KUL,UL521,2024-03-20 14:15:00,FOREIGNER,MATT,KADIR,IND,MALE,O3177391,2038-02-13,IND,NaN,1996-02-20,IND,False,False,False,CMB,KUL,CWVKTW,AVF,2023-02-14,1,MATT,KADIR,MALE,2023-02-14,1,92266022,2023-02-13 18:30:00,CARD,USD,2023-02-13 18:30:00,MATT KADIR,LKA,PAX,VP2023663,2025-07-26,2024-03-20,2025-01-31 14:15:00,2024-03,O3177391_IND,3
4,UL521-2024-03-20T19:45-CMB-KUL,UL521,2024-03-20 14:15:00,FOREIGNER,GABRIEL,BINTI DOLLAH,USA,MALE,G2164424,2039-11-17,USA,NaN,1995-10-05,USA,False,False,False,CMB,KUL,KJKZBS,AVF,2023-01-03,1,GABRIEL,BINTI DOLLAH,MALE,2023-01-03,2,23142285,2023-01-02 18:30:00,CARD,USD,2023-01-02 18:30:00,GABRIEL BINTI DOLLAH,LKA,PAX,VP2023206,2025-08-18,2024-03-20,2025-01-31 14:15:00,2024-03,G2164424_USA,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63971,6E 1805-2024-10-24T20:15-DEL-TAS,6E 1805,2024-10-24 17:30:00,FOREIGNER,FAISAL,BINTI ADNAN,AUS,FEMALE,H2229939,2039-08-03,AUS,NaN,1998-05-18,AUS,False,False,False,DEL,TAS,TKURIR,AVF,2023-01-21,1,FAISAL,BINTI ADNAN,FEMALE,2023-01-21,1,25418633,2023-01-20 18:30:00,CARD,USD,2023-01-20 18:30:00,FAISAL BINTI ADNAN,LKA,PAX,VP2023161,2025-08-16,2024-10-24,2025-01-31 17:30:00,2024-10,H2229939_AUS,107
63972,6E 1805-2024-10-24T20:15-DEL-TAS,6E 1805,2024-10-24 17:30:00,FOREIGNER,DIYA,DARUS,FJI,FEMALE,L2518850,2033-07-14,FJI,NaN,1996-05-14,FJI,False,False,False,DEL,TAS,VBUKBZ,AVF,2023-03-24,1,DIYA,DARUS,FEMALE,2023-03-24,1,66083936,2023-03-23 18:30:00,CARD,USD,2023-03-23 18:30:00,DIYA DARUS,LKA,PAX,VP2023611,2025-05-18,2024-10-24,2025-01-31 17:30:00,2024-10,L2518850_FJI,106
63973,6E 1805-2024-10-24T20:15-DEL-TAS,6E 1805,2024-10-24 17:30:00,FOREIGNER,HANNA,BINTI DOLLAH,USA,MALE,Q1005123,2028-04-27,USA,NaN,2017-02-22,USA,False,False,False,DEL,TAS,EZENUB,AVF,2023-03-18,1,HANNA,BINTI DOLLAH,MALE,2023-03-18,1,48544203,2023-03-17 18:30:00,CARD,USD,2023-03-17 18:30:00,HANNA BINTI DOLLAH,LKA,PAX,VP2023689,2025-07-02,2024-10-24,2025-01-31 17:30:00,2024-10,Q1005123_USA,87
63974,6E 1805-2024-10-24T20:15-DEL-TAS,6E 180

data.to_pickle("data.pkl")

In [5]:
# Example of the specific flight details
specific_flight_ID = 'TK8573-2024-10-24T06:00-IST-TAS'
flight_data = data[(data['flightDetails__id'] == specific_flight_ID)]

<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>04-Feature Engineering</h3> </font>
</div>

The feature_engineering function processes a dataset of flight details to create new features that enhance the model's predictive power. 

It begins by removing duplicate entries based on passenger_id, ensuring each passenger is uniquely represented. Key features are engineered, such as age, calculated from the difference between the passenger's date of birth (dob) and the arrival_date, and several features related to document and visa validity, such as travel_document_validity_days and visa_validity_days. 

Other features include days_since_booking, indicating how long before arrival a booking was made, and boolean flags like multiple_luggage and recently_issued_passport. The function also marks high_risk_nationalities based on predefined high-risk countries.

Travel frequency over different timeframes (1, 3, and 6 months) is computed, and gaps between trips are analyzed, with features like last_trip_gap_days and average_gap_between_trips_days. 

The function flags potential risks such as a mismatch between residenceCountry and nationality, and passengers holding multiple concurrent visas. Finally, all missing values are filled with zeros to ensure data consistency before returning the enriched dataset.

In [6]:
def feature_engineering(flight_data):
    
    # Drop duplicates based on 'passenger_id'
    flight_data = flight_data.drop_duplicates(subset='passenger_id')

    # Feature engineering
    flight_data['age'] = (flight_data['arrival_date'] - flight_data['dob']).dt.days // 365
    flight_data['travel_document_validity_days'] = (flight_data['travelDocument_expiryDate'] - flight_data['arrival_date']).dt.days
    flight_data['visa_validity_days'] = (flight_data['visaExpiryDate'] - flight_data['arrival_date']).dt.days
    flight_data['days_since_booking'] = (flight_data['arrival_date'] - flight_data['bookingDate']).dt.days
    flight_data['multiple_luggage'] = flight_data['travelData_noOfCheckingLuggage'] > 1
    flight_data['payment_to_arrival_gap'] = (flight_data['arrival_date'] - flight_data['paidParty_paidDate']).dt.days
    flight_data['recently_issued_passport'] = (flight_data['arrival_date'] - flight_data['travelDocument_expiryDate']).dt.days < 365
    high_risk_nationalities = ['TWN', 'PAK', 'FJI']
    flight_data['high_risk_nationalities'] = flight_data['travelDocument_issueCountry'].isin(high_risk_nationalities)

    max_arrival_date = flight_data['arrival_date'].max()
    timeframes = {
        'last_1_month': max_arrival_date - pd.DateOffset(months=1),
        'last_3_months': max_arrival_date - pd.DateOffset(months=3),
        'last_6_months': max_arrival_date - pd.DateOffset(months=6),
    }

    travel_counts = []
    for period_name, period_start in timeframes.items():
        count = (flight_data[flight_data['arrival_date'] >= period_start]
                 .groupby(['travelDocument_docNo', 'nationality'])['arrival_date']
                 .count().reset_index())
        count.columns = ['travelDocument_docNo', 'nationality', period_name]
        travel_counts.append(count)

    travel_counts_df = travel_counts[0]
    for count in travel_counts[1:]:
        travel_counts_df = travel_counts_df.merge(count, on=['travelDocument_docNo', 'nationality'], how='outer')

    flight_data = flight_data.merge(travel_counts_df, on=['travelDocument_docNo', 'nationality'], how='left')
    flight_data['last_trip_gap_days'] = flight_data.groupby(['travelDocument_docNo', 'nationality'])['arrival_date'].diff().dt.days
    flight_data['average_gap_between_trips_days'] = flight_data.groupby(['travelDocument_docNo', 'nationality'])['last_trip_gap_days'].transform('mean')
    flight_data['residence_vs_nationality_mismatch'] = flight_data['nationality'] != flight_data['residenceCountry']
    flight_data['multiple_concurrent_visas'] = flight_data.groupby(['travelDocument_docNo', 'nationality'])['visaNo'].transform('nunique') > 1

    # Replace all NaN values with zero
    flight_data.fillna(0, inplace=True)
    
    return flight_data


In [7]:
featured_flight_data= feature_engineering(flight_data)
featured_flight_data

,flightDetails__id,flightDetails_flightNumber,flightDetails_arrivalTime,type,givenName,lastName,nationality,gender,travelDocument_docNo,travelDocument_expiryDate,travelDocument_issueCountry,firstTransitPorts,dob,residenceCountry,dependent,crew,transit,portOfEmbark,portOfDisembark,travellerReferenceId,travellerReferenceType,bookingDate,paxCount,reservation_firstName,reservation_lastName,reservation_gender,travelData_reservationDate,travelData_noOfCheckingLuggage,paidParty_cardNumber,paidParty_expirationDate,paidParty_cardType,paidParty_paidCurrency,paidParty_paidDate,paidParty_cardHolderName,paidParty_payCountry,travelData_paxTypes,visaNo,visaExpiryDate,arrival_date,arrival_time,arrival_month_year,unique_id,passenger_id,age,travel_document_validity_days,visa_validity_days,days_since_booking,multiple_luggage,payment_to_arrival_gap,recently_issued_passport,high_risk_nationalities,last_1_month,last_3_months,last_6_months,last_trip_gap_days,average_gap_between_trips_days,residence_vs_nationality_mismatch,multiple_concurrent_visas
0,TK8573-2024-10-24T06:00-IST-TAS,TK8573,2024-10-24 07:00:00,FOREIGNER,GEORG,AHMAD,LKA,MALE,C1628526,2034-10-11,LKA,0,2014-09-13,LKA,False,False,False,IST,TAS,PYLHID,AVF,2023-02-02,1,GEORG,AHMAD,MALE,2023-02-02,3,73458057,2023-02-01 18:30:00,CARD,USD,2023-02-01 18:30:00,GEORG AHMAD,LKA,PAX,VP2023341,2025-05-28,2024-10-24,2025-01-31 07:00:00,2024-10,C1628526_LKA,339,10,3639,216,630,True,630,True,False,1,1,1,0.0,0.0,False,False
1,TK8573-2024-10-24T06:00-IST-TAS,TK8573,2024-10-24 07:00:00,FOREIGNER,ERIC,TING,AUS,MALE,R3429075,2024-07-26,AUS,0,2003-10-05,AUS,False,False,False,IST,TAS,DEYIMO,AVF,2023-03-11,1,ERIC,TING,MALE,2023-03-11,1,20732513,2023-03-10 18:30:00,CARD,USD,2023-03-10 18:30:00,ERIC TING,LKA,PAX,VP2023310,2025-07-04,2024-10-24,2025-01-31 07:00:00,2024-10,R3429075_AUS,143,21,-90,253,593,False,593,True,False,1,1,1,0.0,0.0,False,False
2,TK8573-2024-10-24T06:00-IST-TAS,TK8573,2024-10-24 07:00:00,FOREIGNER,IMRAN,ARIFF,CHN,MALE,U3180739,2032-12-13,CHN,0,2000-12-04,CHN,False,False,False,IST,TAS,OSKUYN,AVF,2023-01-27,1,IMRAN,ARIFF,MALE,2023-01-27,3,70792052,2023-01-26 18:30:00,CARD,USD,2023-01-26 18:30:00,IMRAN ARIFF,LKA,PAX,VP2023033,2025-05-31,2024-10-24,2025-01-31 07:00:00,2024-10,U3180739_CHN,338,23,2972,219,636,True,636,True,False,1,1,1,0.0,0.0,False,False
3,TK8573-2024-10-24T06:00-IST-TAS,TK8573,2024-10-24 07:00:00,FOREIGNER,CHARLES,TING,AUS,FEMALE,W3136724,2029-09-17,AUS,0,1982-12-21,AUS,False,False,False,IST,TAS,XHTDRC,AVF,2023-02-21,1,CHARLES,TING,FEMALE,2023-02-21,1,43455304,2023-02-20 18:30:00,CARD,USD,2023-02-20 18:30:00,CHARLES TING,LKA,PAX,VP2023147,2025-08-28,2024-10-24,2025-01-31 07:00:00,2024-10,W3136724_AUS,69,41,1789,308,611,False,611,True,False,1,1,1,0.0,0.0,False,False
4,TK8573-2024-10-24T06:00-IST-TAS,TK8573,2024-10-24 07:00:00,FOREIGNER,HANNA,KADIR,PAK,MALE,W1280371,2033-03-12,PAK,0,2005-09-02,PAK,False,False,False,IST,TAS,BOBFKK,AVF,2023-02-18,1,HANNA,KADIR,MALE,2023-02-18,2,66594323,2023-02-17 18:30:00,CARD,USD,2023-02-17 18:30:00,HANNA KADIR,LKA,PAX,VP2023131,2025-07-31,2024-10-24,2025-01-31 07:00:00,2024-10,W1280371_PAK,145,19,3061,280,614,True,614,True,True,1,1,1,0.0,0.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,TK8573-2024-10-24T06:00-IST-TAS,TK8573,2024-10-24 07:00:00,FOREIGNER,HANNA,KADIR,GBR,MALE,B2247042,2026-04-22,GBR,0,1975-06-09,GBR,False,False,False,IST,TAS,BDVEHV,AVF,2023-03-10,1,HANNA,KADIR,MALE,2023-03-10,2,25331419,2023-03-09 18:30:00,CARD,USD,2023-03-09 18:30:00,HANNA KADIR,LKA,PAX,VP2023229,2025-06-07,2024-10-24,2025-01-31 07:00:00,2024-10,B2247042_GBR,113,49,545,226,594,True,594,True,False,1,1,1,0.0,0.0,False,False
324,TK8573-2024-10-24T06:00-IST-TAS,TK8573,2024-10-24 07:00:00,FOREIGNER,DIYA,LEO,IND,MALE,G3885147,2039-12-17,IND,0,2019-07-17,IND,False,Fals

<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>05- Data Encoding</h3> </font>
</div>

The encode_flight_data function prepares and encodes a dataset for further analysis or machine learning tasks.

It begins by ensuring the paxCount column, representing passenger count, is numeric, converting any non-numeric entries to NaN and subsequently filling these NaN values with 0. 

It checks for the existence of the gender column, encoding it into dummy variables (gender_FEMALE and gender_MALE) and then removing the original gender column.

The function identifies specific boolean columns related to passenger risk factors, such as multiple_luggage and recently_issued_passport, ensuring these columns are converted to integer format. 

It selects a predefined list of relevant columns for the final dataset, filtering out any that do not exist in the current dataset to avoid errors. The passenger_id column is set as the index for the DataFrame, facilitating efficient data handling. 

In [8]:
def encode_flight_data(featured_flight_data):
    
    # Ensure 'paxCount' is numeric, coercing errors to NaN
    featured_flight_data['paxCount'] = pd.to_numeric(featured_flight_data['paxCount'], errors='coerce')
    
    # Fill NaN values in 'paxCount' with 0
    featured_flight_data['paxCount'].fillna(0, inplace=True)
    
    # Check if 'gender' column exists before encoding
    if 'gender' in featured_flight_data.columns:
        gender_dummies = pd.get_dummies(featured_flight_data['gender'], prefix='gender', drop_first=False)
        featured_flight_data = pd.concat([featured_flight_data, gender_dummies], axis=1)
        featured_flight_data.drop(columns=['gender'], inplace=True)

    # Define boolean columns and check their existence
    boolean_columns = [
        'multiple_luggage', 'recently_issued_passport', 'high_risk_nationalities', 
        'residence_vs_nationality_mismatch', 'multiple_concurrent_visas', 
    ]
    existing_boolean_columns = [col for col in boolean_columns if col in featured_flight_data.columns]
    featured_flight_data[existing_boolean_columns] = featured_flight_data[existing_boolean_columns].astype(int)

    # Select final columns for the DataFrame
    selected_columns = [
        'passenger_id', 'paxCount', 'age', 
        'travel_document_validity_days', 'visa_validity_days', 'days_since_booking', 
        'payment_to_arrival_gap', 'last_1_month', 'last_3_months', 'last_6_months', 
        'last_trip_gap_days', 'average_gap_between_trips_days',
        'multiple_luggage', 'recently_issued_passport', 'high_risk_nationalities', 
        'residence_vs_nationality_mismatch', 'multiple_concurrent_visas',
        'gender_FEMALE', 'gender_MALE'
    ]
    # Filter existing columns
    selected_columns = [col for col in selected_columns if col in featured_flight_data.columns]
    
    featured_flight_data = featured_flight_data[selected_columns]
    
    # Set 'passenger_id' as the index
    featured_flight_data.set_index('passenger_id', inplace=True)

    return featured_flight_data

In [9]:
encoded_flight_data = encode_flight_data(featured_flight_data)
encoded_flight_data

,paxCount,age,travel_document_validity_days,visa_validity_days,days_since_booking,payment_to_arrival_gap,last_1_month,last_3_months,last_6_months,last_trip_gap_days,average_gap_between_trips_days,multiple_luggage,recently_issued_passport,high_risk_nationalities,residence_vs_nationality_mismatch,multiple_concurrent_visas,gender_FEMALE,gender_MALE
passenger_id,,,,,,,,,,,,,,,,,,
339,1,10,3639,216,630,630,1,1,1,0.0,0.0,1,1,0,0,0,False,True
143,1,21,-90,253,593,593,1,1,1,0.0,0.0,0,1,0,0,0,False,True
338,1,23,2972,219,636,636,1,1,1,0.0,0.0,1,1,0,0,0,False,True
69,1,41,1789,308,611,611,1,1,1,0.0,0.0,0,1,0,0,0,True,False
145,1,19,3061,280,614,614,1,1,1,0.0,0.0,1,1,1,0,0,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,1,49,545,226,594,594,1,1,1,0.0,0.0,1,1,0,0,0,False,True
111,1,5,5532,267,590,590,1,1,1,0.0,0.0,0,1,0,0,0,False,True
110,1,26,4285,258,631,631,1,1,1,0.0,0.0,1,1,0,0,0,True,False


<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>06-Clustering & Anomaly Detection</h3> </font>
</div>


The function clustering_and_anomaly_detection performs a comprehensive analysis to identify anomalies in encoded flight data using clustering and anomaly detection techniques. 

It starts by selecting key features such as age, travel document validity, and travel gaps, which are standardized using StandardScaler. K-Means clustering is applied to group passengers into clusters based on these features.

The function then uses Isolation Forest to detect anomalies within these clusters, with predictions and scores added to the dataset. To enhance anomaly detection, an Autoencoder model is trained on the data, and reconstruction errors are calculated for each instance. 

A custom threshold is established for each cluster based on reconstruction errors to classify anomalies. The results from Isolation Forest and the Autoencoder are combined into a final combined_anomaly column, with a final_anomaly_score representing the maximum anomaly score. 

The function returns the dataset rows flagged as anomalies, providing a robust framework for detecting deviations in passenger behavior.

In [10]:
def clustering_and_anomaly_detection(encoded_flight_data, n_clusters=4, random_state=42, contamination=0.05, epochs=50, batch_size=32):
    
    # Define features for anomaly detection
    features = [
        'age', 'paxCount', 'multiple_luggage', 'travel_document_validity_days', 'visa_validity_days',
        'days_since_booking', 'payment_to_arrival_gap', 'last_1_month', 'last_3_months', 'last_6_months',
        'last_trip_gap_days', 'average_gap_between_trips_days', 'gender_FEMALE', 'gender_MALE',
        'recently_issued_passport', 'high_risk_nationalities', 'residence_vs_nationality_mismatch', 
        'multiple_concurrent_visas'
    ]
    
    # Select anomaly data based on defined features
    anomaly_data = encoded_flight_data[features].copy()
    
    # Define numerical columns for scaling
    numerical_columns = [
        'age', 'travel_document_validity_days', 'visa_validity_days', 
        'days_since_booking', 'payment_to_arrival_gap', 'last_1_month', 
        'last_3_months', 'last_6_months', 'last_trip_gap_days', 
        'average_gap_between_trips_days'
    ]
    
    # Define clustering features
    clustering_features = [
        'age', 'paxCount', 'travel_document_validity_days', 'visa_validity_days',
        'days_since_booking', 'payment_to_arrival_gap', 'last_1_month', 
        'last_3_months', 'last_6_months', 'last_trip_gap_days', 'multiple_luggage',
        'average_gap_between_trips_days', 'high_risk_nationalities', 
        'residence_vs_nationality_mismatch', 'multiple_concurrent_visas', 
        'recently_issued_passport', 'gender_FEMALE', 'gender_MALE'
    ]
    
    # Initialize the scaler
    scaler = StandardScaler()
    
    # Fit and transform the numerical data
    anomaly_data[numerical_columns] = scaler.fit_transform(anomaly_data[numerical_columns])
    
    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    anomaly_data['flight_cluster'] = kmeans.fit_predict(anomaly_data[clustering_features])

    # Define the features to be used for anomaly detection
    anomaly_features = [
        'age', 'paxCount', 'multiple_luggage', 'travel_document_validity_days', 'visa_validity_days',
        'days_since_booking', 'payment_to_arrival_gap', 'last_1_month', 'last_3_months', 'last_6_months',
        'last_trip_gap_days', 'average_gap_between_trips_days', 'gender_FEMALE', 'gender_MALE',
        'recently_issued_passport', 'high_risk_nationalities', 'residence_vs_nationality_mismatch', 
        'multiple_concurrent_visas', 'flight_cluster'
    ]

    # Select the relevant features for anomaly detection
    scaled_data = anomaly_data[anomaly_features]

    # Initialize the Isolation Forest model
    iso_forest = IsolationForest(contamination=contamination, random_state=42)  

    # Fit the model to the data
    iso_forest.fit(scaled_data)

    # Predict anomalies (1 = normal, -1 = anomaly)
    anomaly_predictions = iso_forest.predict(scaled_data)

    # Anomaly score for each point (higher score indicates more anomalous)
    anomaly_scores = iso_forest.decision_function(scaled_data)

    # Add anomaly predictions and scores to the dataset
    anomaly_data['anomaly_prediction'] = anomaly_predictions
    anomaly_data['anomaly_score'] = anomaly_scores

    # Define Autoencoder Model
    input_dim = scaled_data.shape[1]  # Number of features

    input_layer = Input(shape=(input_dim,))
    encoded = Dense(32, activation='relu')(input_layer)
    decoded = Dense(input_dim, activation='sigmoid')(encoded)

    autoencoder = Model(input_layer, decoded)

    # Compile and fit the model
    autoencoder.compile(optimizer=Adam(), loss='mse')
    autoencoder.fit(scaled_data, scaled_data, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)

    # Get reconstruction error
    reconstructed = autoencoder.predict(scaled_data)
    reconstruction_error = np.mean(np.abs(scaled_data - reconstructed), axis=1)
    
    # Add reconstruction errors to the dataset
    anomaly_data['reconstruction_error'] = reconstruction_error

    # Define thresholds for each cluster based on reconstruction error (mean + 1.5 * std deviation)
    thresholds = anomaly_data.groupby('flight_cluster')['reconstruction_error'].agg(['mean', 'std'])
    thresholds['custom_threshold'] = thresholds['mean'] + 1.5 * thresholds['std']

    # Apply the threshold to each cluster
    def apply_threshold(row, thresholds):
        cluster = row['flight_cluster']
        return row['reconstruction_error'] > thresholds.loc[cluster, 'custom_threshold']

    anomaly_data['autoencoder_anomaly'] = anomaly_data.apply(apply_threshold, axis=1, thresholds=thresholds)

    # Combine the anomaly results into a final 'combined_anomaly' column
    anomaly_data['combined_anomaly'] = ((anomaly_data['anomaly_prediction'] == -1) | (anomaly_data['autoencoder_anomaly'] == 1)).astype(int)

    # Final anomaly score (use the maximum of the anomaly score and reconstruction error)
    anomaly_data['final_anomaly_score'] = np.maximum(anomaly_data['anomaly_score'], anomaly_data['reconstruction_error'])

    # Final anomalies output
    anomalies = anomaly_data[anomaly_data['combined_anomaly'] == 1][anomaly_features]

    # Reset index to convert passenger_id from index to a column in clustered_anomaly_data
    clustered_anomaly_data = anomalies.reset_index()

    # Perform the merge on passenger_id
    merged_arm_data = encoded_flight_data.merge(
        clustered_anomaly_data[['passenger_id']],  # Keep only passenger_id column
        on='passenger_id',
        how='inner'
    )

    # Set passenger_id as the index in the merged dataframe
    merged_arm_data.set_index('passenger_id', inplace=True)

    # Define the thresholds for binarization
    thresholds = {
        'age': 18,
        'paxCount': 2,
        'travel_document_validity_days': 30,
        'visa_validity_days': 30,
        'days_since_booking': 7,
        'payment_to_arrival_gap': 7,
        'last_trip_gap_days': 30,
        'average_gap_between_trips_days': 30
    }

    # Binarize features based on thresholds
    merged_arm_data['is_senior'] = (merged_arm_data['age'] >= thresholds['age']).astype(int)
    merged_arm_data['is_group_travel'] = (merged_arm_data['paxCount'] >= thresholds['paxCount']).astype(int)
    merged_arm_data['is_travel_document_expiring'] = (merged_arm_data['travel_document_validity_days'] < thresholds['travel_document_validity_days']).astype(int)
    merged_arm_data['is_visa_expiring'] = (merged_arm_data['visa_validity_days'] < thresholds['visa_validity_days']).astype(int)
    merged_arm_data['is_last_minute_booking'] = (merged_arm_data['days_since_booking'] < thresholds['days_since_booking']).astype(int)
    merged_arm_data['is_last_minute_payment'] = (merged_arm_data['payment_to_arrival_gap'] < thresholds['payment_to_arrival_gap']).astype(int)
    merged_arm_data['is_frequent_travel'] = (merged_arm_data['last_trip_gap_days'] < thresholds['average_gap_between_trips_days']).astype(int)
    merged_arm_data['is_high_avg_freq_travel'] = (merged_arm_data['average_gap_between_trips_days'] < thresholds['average_gap_between_trips_days']).astype(int)

    # Define thresholds for last_n_months features
    months_thresholds = {
        'last_1_month': 3,  # 3 trips or more in the last month could be a red flag
        'last_3_months': 5,  # 5 trips or more in the last 3 months could indicate a pattern shift
        'last_6_months': 7   # 7 trips or more in the last 6 months might indicate frequent travel
    }

    # Binarize last n months features
    merged_arm_data['is_frequent_travel_last_1_month'] = (merged_arm_data['last_1_month'] >= months_thresholds['last_1_month']).astype(int)
    merged_arm_data['is_frequent_travel_last_3_month'] = (merged_arm_data['last_3_months'] >= months_thresholds['last_3_months']).astype(int)
    merged_arm_data['is_frequent_travel_last_6_month'] = (merged_arm_data['last_6_months'] >= months_thresholds['last_6_months']).astype(int)

    # Drop original columns after binarization
    merged_arm_data = merged_arm_data.drop(columns=['age', 'paxCount', 'travel_document_validity_days',
                                                   'visa_validity_days', 'days_since_booking', 'payment_to_arrival_gap',
                                                   'last_1_month', 'last_3_months', 'last_6_months', 'last_trip_gap_days',
                                                   'average_gap_between_trips_days'])

    # Convert all columns to boolean (True/False) for optimal performance in mlxtend
    merged_arm_data = merged_arm_data.astype(bool)

    return merged_arm_data

In [11]:
anomaly_data = clustering_and_anomaly_detection(encoded_flight_data)
anomaly_data

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - loss: 0.6111 - val_loss: 0.5488
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.6054 - val_loss: 0.5210
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.6057 - val_loss: 0.4938
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.5479 - val_loss: 0.4666
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.5450 - val_loss: 0.4390
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.5411 - val_loss: 0.4117
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.4887 - val_loss: 0.3856
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.4112 - val_loss: 0.3611
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.4049 - val_loss: 0.3394
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.3912 - val_loss: 0.3213
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.3473 - val_loss: 0.3066
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.3

,multiple_luggage,recently_issued_passport,high_risk_nationalities,residence_vs_nationality_mismatch,multiple_concurrent_visas,gender_FEMALE,gender_MALE,is_senior,is_group_travel,is_travel_document_expiring,is_visa_expiring,is_last_minute_booking,is_last_minute_payment,is_frequent_travel,is_high_avg_freq_travel,is_frequent_travel_last_1_month,is_frequent_travel_last_3_month,is_frequent_travel_last_6_month
passenger_id,,,,,,,,,,,,,,,,,,
221,False,True,False,False,False,False,True,False,False,False,False,False,False,True,True,False,False,False
198,True,True,True,False,False,False,True,True,False,False,False,False,False,True,True,False,False,False
209,False,True,True,False,False,True,False,False,False,False,False,False,False,True,True,False,False,False
210,True,True,False,False,False,True,False,False,False,True,False,False,False,True,True,False,False,False
229,True,True,False,False,False,True,False,False,False,False,False,False,False,True,True,False,False,False
302,True,True,False,False,False,False,True,True,False,False,False,False,False,True,True,False,False,False
315,False,True,True,False,False,False,True,False,False,True,False,False,False,True,True,False,False,False
321,False,True,True,False,False,True,False,False,False,False,False,False,False,True,True,False,False,False
310,False,True,True,False,False,True,False,True,False,False,False,False,False,True,True,False,False,False


<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>07-Association Rule Mining</h3> </font>
</div>

The function association_rule_mining processes and analyzes flight data to identify high-risk passengers based on encoded flight features and clustered anomaly data.

It merges these datasets, applies threshold-based binarization to several features such as age, travel document validity, and booking gaps, and then uses the FP-Growth algorithm to mine frequent itemsets.

It generates association rules, sorts them by confidence and lift, and selects three distinct rules while ensuring that no overlapping features are used within each rule.

These rules are converted into human-readable sentences and applied to flag passengers who meet the criteria, with flagged passengers grouped and merged with selected flight data to produce a final dataset of high-risk individuals, complete with their details and the specific rules they triggered.

In [12]:
def association_rule_mining(encoded_flight_data, anomaly_data):
    
    # Apply the FP-Growth algorithm
    frequent_itemsets = fpgrowth(anomaly_data, min_support=0.01, use_colnames=True)

    # Generate association rules using lift as the metric
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

    # Sort rules by confidence and lift
    sorted_rules = rules.sort_values(by=['confidence', 'lift'], ascending=False)

    # Initialize variables to track distinct rules and used features
    used_features = set()
    distinct_rules = []

    # Define travel count features to ensure they do not appear together
    travel_count_features = {'is_frequent_travel_last_1_month', 'is_frequent_travel_last_3_month', 'is_frequent_travel_last_6_month',
                             'is_frequent_travel', 'is_high_avg_freq_travel'}

    # Iterate through sorted rules to find three distinct rules
    for _, rule in sorted_rules.iterrows():
        antecedents = set(rule['antecedents'])
        consequents = set(rule['consequents'])

        # Combine antecedents and consequents to check for travel count features
        all_features = antecedents | consequents

        # Check if any of the travel count features appear together
        if len(all_features & travel_count_features) > 1:
            continue  # Skip if more than one travel count feature is present

        # Ensure no overlap with used features
        if not (antecedents & used_features) and not (consequents & used_features):
            distinct_rules.append(rule)
            used_features.update(all_features)

        if len(distinct_rules) == 3:
            break

    # Convert the distinct rules into a DataFrame
    distinct_rules_df = pd.DataFrame(distinct_rules)
    
    return distinct_rules_df

In [13]:
rules_df = association_rule_mining(encoded_flight_data, anomaly_data)
rules_df

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
7333,"(gender_FEMALE, is_group_travel)",(is_visa_expiring),0.028571,0.028571,0.028571,1.0,35.000000,1.0,0.027755,inf,1.000000,1.000000,1.0,1.000000
3529,"(is_senior, is_travel_document_expiring)","(high_risk_nationalities, multiple_luggage)",0.028571,0.171429,0.028571,1.0,5.833333,1.0,0.023673,inf,0.852941,0.166667,1.0,0.583333
2,(recently_issued_passport),(is_frequent_travel),1.000000,1.000000,1.000000,1.0,1.000000,1.0,0.000000,inf,0.000000,1.000000,0.0,1.000000


<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>08-Flagged Passenger Details Identification</h3> </font>
</div>

The flagged_passenger_details function is designed to identify high-risk passengers based on predefined association rules and passenger features, such as luggage type, passport status, nationality, and travel behavior.

It first maps these features to human-readable descriptions to generate clear rule statements. The function then applies these rules to a dataset of passengers, flagging those who meet the conditions specified by the rules. 

Each flagged passenger is linked to the corresponding rule numbers, indicating which conditions they met. After aggregating this information, the function merges it with personal flight data, such as the passenger's name, date of birth, nationality, and travel document details. The final output includes a list of human-readable rule statements and a DataFrame with detailed passenger information, along with the rule numbers that flagged them as high-risk.


In [14]:
def flagged_passenger_details(rules_df, anomaly_data):

    # Feature map for human-readable feature names
    feature_map = {
        'multiple_luggage': 'carry multiple pieces of luggage',
        'recently_issued_passport': 'have a recently issued passport',
        'high_risk_nationalities': 'belong to a high-risk nationality',
        'residence_vs_nationality_mismatch': 'have a mismatch between residence country and nationality',
        'multiple_concurrent_visas': 'hold multiple concurrent visas',
        'gender_FEMALE': 'are female',
        'gender_MALE': 'are male',
        'is_senior': 'are an adult (above 18)',
        'is_group_travel': 'are part of a group travel',
        'is_travel_document_expiring': 'have a travel document with less than 30 days of validity',
        'is_visa_expiring': 'have a visa with less than 30 days of validity',
        'is_last_minute_booking': 'booked travel less than 7 days before the trip',
        'is_last_minute_payment': 'made payment less than 7 days before the trip',
        'is_frequent_travel': 'with a last trip gap of less than 30 days',
        'is_high_avg_freq_travel': 'with an average historical travel gap of less than 30 days',
        'is_frequent_travel_last_1_month': 'traveled 3 or more times in the last month',
        'is_frequent_travel_last_3_month': 'traveled 5 or more times in the last 3 months',
        'is_frequent_travel_last_6_month': 'traveled 7 or more times in the last 6 months'
    }
    # Generate readable rule sentences
    sentences = []
    
    # Initialize the counter for rule numbering
    rule_number = 1
    
    for _, row in rules_df.iterrows():
        # Extract antecedents and consequents as frozensets
        antecedents = row['antecedents']
        consequents = row['consequents']
        
        # Ensure antecedents and consequents are in a list form (converting frozenset if needed)
        if isinstance(antecedents, frozenset):
            antecedents = list(antecedents)
        if isinstance(consequents, frozenset):
            consequents = list(consequents)
        
        # Convert antecedents and consequents to readable conditions
        antecedent_conditions = [feature_map.get(str(antecedent), antecedent) for antecedent in antecedents]
        consequent_conditions = [feature_map.get(str(consequent), consequent) for consequent in consequents]
        
        # Create the rule sentence with custom numbering
        sentence = f"Rule {rule_number}: Passengers who " + ", ".join(antecedent_conditions + consequent_conditions) + " are identified as high-risk passengers."
        sentences.append(sentence)

        # Increment the rule number for the next rule
        rule_number += 1

    # Ensure 'antecedents' and 'consequents' are in a proper format for filtering
    rules_df['antecedents'] = rules_df['antecedents'].apply(lambda x: list(x))
    rules_df['consequents'] = rules_df['consequents'].apply(lambda x: list(x))

    # Initialize a list to store flagged passengers
    flagged_passengers = []

    # Initialize a rule counter (assuming the numbering starts from 1)
    rule_number = 1

    # Iterate over each rule to filter passengers
    for _, rule in rules_df.iterrows():
        # Combine antecedents and consequents for filtering
        conditions = rule['antecedents'] + rule['consequents']
        
        # Filter passengers who meet all conditions in the current rule
        mask = anomaly_data[conditions].all(axis=1)
        flagged_passenger_ids = anomaly_data[mask].index.tolist()
        
        # Append passenger IDs and their corresponding rule number to the list
        for pid in flagged_passenger_ids:
            flagged_passengers.append({
                'passenger_id': pid,
                'rule_number': rule_number  # Append the rule number
            })

        # Increment the rule number for the next rule
        rule_number += 1

    # Convert the list of flagged passengers into a DataFrame
    flagged_passengers_df = pd.DataFrame(flagged_passengers)

    # Group flagged passengers by passenger_id and aggregate rule numbers into a comma-separated string
    flagged_passengers_grouped = flagged_passengers_df.groupby('passenger_id')['rule_number'].apply(lambda x: ', '.join(map(str, x))).reset_index()
    
    # Rename the aggregated column for clarity
    flagged_passengers_grouped.rename(columns={'rule_number': 'Flagged_Rule_Numbers'}, inplace=True)
    
    # Select the required columns from the 'featured_flight_data' dataframe
    selected_columns = ['givenName', 'lastName', 'dob', 'gender', 'nationality', 'travelDocument_docNo', 'visaNo', 'residenceCountry', 'passenger_id']
    data_selected = flight_data[selected_columns]
    
    # Merge 'flagged_passengers_grouped' with 'data_selected' on passenger_id
    final_merged_df = data_selected.merge(flagged_passengers_grouped, on='passenger_id', how='inner')
    
    # Rename the columns (modify the new names as per your requirement)
    final_merged_df = final_merged_df.rename(columns={
    'givenName': 'Given_Name',
    'lastName': 'Last_Name',
    'dob': 'Date_of_Birth',
    'gender': 'Gender',
    'nationality': 'Nationality',
    'travelDocument_docNo': 'Travel_Document_Number',
    'visaNo': 'Visa_Number',
    'residenceCountry': 'Residence_Country'})

    return sentences,final_merged_df


In [15]:
sentences,final_merged_df= flagged_passenger_details(rules_df, anomaly_data)
sentences
final_merged_df

['Rule 1: Passengers who are female, are part of a group travel, have a visa with less than 30 days of validity are identified as high-risk passengers.',
 'Rule 2: Passengers who are an adult (above 18), have a travel document with less than 30 days of validity, belong to a high-risk nationality, carry multiple pieces of luggage are identified as high-risk passengers.',
 'Rule 3: Passengers who have a recently issued passport, with a last trip gap of less than 30 days are identified as high-risk passengers.']

,Given_Name,Last_Name,Date_of_Birth,Gender,Nationality,Travel_Document_Number,Visa_Number,Residence_Country,passenger_id,Flagged_Rule_Numbers
0,FAISAL,BIN ABD WAHAB,2018-10-17,MALE,GBR,S2121183,VP2023196,GBR,221,3
1,ROBERT,BIN ABAS,1976-08-10,MALE,FJI,U1266074,VP2023961,FJI,198,3
2,JOHN,BIN DARUS,2014-08-11,FEMALE,FJI,I2711777,VP2023740,FJI,209,3
3,LILY,KADIR,2009-10-21,FEMALE,JPN,P1959301,VP2023378,JPN,210,3
4,LILY,AHMAD,2013-06-08,FEMALE,USA,J1215251,VP2023945,USA,229,3
5,GEORG,OOI,2005-11-27,MALE,USA,D2014234,VP2023677,USA,302,3
6,HARUN,BIN ABD WAHAB,2007-01-17,MALE,FJI,Q3417662,VP2023155,FJI,315,3
7,HANNA,ARIFF,2013-06-12,FEMALE,FJI,S1940314,VP2023647,FJI,321,3
8,HARUN,BINTI MAMAT,1975-04-04,FEMALE,PAK,F1926624,VP2023360,PAK,310,3
9,IMRAN,BINTI DOLLAH,1996-12-04,FEMALE,PAK,E2195543,VP2023531,PAK,307,"2, 3"


<div style="text-align: left;">
    <h3> </font> <font color = #FFA500>08-Reinforcement Learning Implimentation (RL Model)</h3> </font>
</div>

The following code defines a custom environment for Reinforcement Learning using OpenAI's Gym library, where an agent interacts with the environment by selecting actions based on predefined association rules. 

The CustomEnv class initializes an environment with a discrete action space (3 actions) and a discrete observation space (10 observations). The environment simulates rule application and calculates rewards for each action through methods like apply_rule_1(), apply_rule_2(), and apply_rule_3(). The environment progresses by taking steps until it reaches a predefined number of steps (10 in this case). 

The  also initializes a Proximal Policy Optimization (PPO) model with specific hyperparameters (e.g., learning rate, number of epochs) to train the agent over 10,000 timesteps. After training, the model can be evaluated by running the environment for 10 steps and predicting actions to interact with the environment. This setup is useful for testing how reinforcement learning can optimize decision-making in a rule-based environment.

In [16]:
# Custom Environment
class CustomEnv(gym.Env):
    def __init__(self, rules_df):
        super(CustomEnv, self).__init__()
        
        # Define action space and observation space
        self.action_space = spaces.Discrete(3)  # Example: 3 actions
        self.observation_space = spaces.Discrete(10)  # Example: 10 different observations (modify as per your case)
        
        # Store rules
        self.rules_df = rules_df  # This is where your association rules dataframe is passed
        self.current_step = 0

    def reset(self):
        self.current_step = 0
        # Return an initial state
        return self.current_step  # Example: initial state is just the current_step
    
    def step(self, action):
        # Example step logic based on your rules and actions
        # Action interpretation depends on your custom logic
        reward = 0
        
        # Perform action and calculate reward
        if action == 0:
            reward = self.apply_rule_1()  # Dummy method to simulate applying a rule
        elif action == 1:
            reward = self.apply_rule_2()  # Dummy method to simulate applying a rule
        else:
            reward = self.apply_rule_3()  # Dummy method to simulate applying a rule

        # Proceed to next state
        self.current_step += 1
        done = self.current_step >= 10  # Example: end after 10 steps
        
        return self.current_step, reward, done, {}

    def apply_rule_1(self):
        # Example of how you would use the association rules for rewards
        # Look up some rules, and compute reward (just a dummy implementation)
        return 1
    
    def apply_rule_2(self):
        # Another rule application logic
        return 2
    
    def apply_rule_3(self):
        # Another rule application logic
        return -1

The  following training logs show how the Proximal Policy Optimization (PPO) model is performing during its learning process across several iterations. 

- **Loss:** This is the overall loss computed during training, which combines several components such as the policy loss and the value loss. It gives an indication of how well the model is learning, with a lower loss typically suggesting better performance. In the logs, the loss is gradually fluctuating between values such as 2.38, 2.46, 2.6, and 3.3. These fluctuations are normal and can be a result of the training dynamics, such as the exploration-exploitation trade-off, changes in the policy, and the difficulty of the environment. The goal is for the model to converge to a low and stable loss over time.

- **Value Loss:** This refers to the difference between the predicted state-value function (how good a state is, based on the model's prediction) and the actual returns observed during training. Value loss is a crucial metric in reinforcement learning, as it directly affects how well the agent can estimate the return (reward) from a given state. The value loss is showing values such as 7.87, 5.93, 6.21, and 5.27 over time, indicating that the model is trying to refine its estimation of state values. These are relatively high, suggesting that the model is still adjusting its value function, but this is expected early in training as the agent learns the environment.

What to make of these numbers:

- Fluctuations in Loss: PPO can exhibit some oscillations in the loss early in training, especially in complex environments. As the model learns, it will try to refine the policy, and the loss should ideally stabilize over time.

- Decreasing Value Loss: The value loss is gradually decreasing (from 7.87 down to 5.27), which is a good sign. It suggests that the model is starting to make better predictions about the value of states, although this process can take time to converge.

In [17]:
# Initialize the custom environment with the mined rules
env = CustomEnv(rules_df)
print(f"Original Action Space: {env.action_space}")
env = DummyVecEnv([lambda: env])

# Initialize PPO model
ppo_model = PPO(
    "MlpPolicy", 
    env, 
    learning_rate=0.0005, 
    n_steps=2048, 
    ent_coef=0.01, 
    clip_range=0.1, 
    n_epochs=10, 
    verbose=1
)
ppo_model.learn(total_timesteps=10000)

# To save the model
ppo_model.save("ppo_custom_env_model")

# To evaluate the model
obs = env.reset()
for _ in range(10):  # Run for 10 steps
    action, _state = ppo_model.predict(obs)
    obs, reward, done, info = env.step(action)
    if done:
        break


Original Action Space: Discrete(3)
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 490  |
|    iterations      | 1    |
|    time_elapsed    | 4    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 340         |
|    iterations           | 2           |
|    time_elapsed         | 12          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008382019 |
|    clip_fraction        | 0.563       |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.09       |
|    explained_variance   | 0.0101      |
|    learning_rate        | 0.0005      |
|    loss                 | 3.21        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0379     |
|    value_loss           | 7.71        |
------------------------